# Data Vortex — Phase 2: SQL Challenge 5
## Monthly Publishing Volume & Trends

### 1. Challenge Description
Analyze monthly publishing volume, month-over-month (MoM) change, percentage change, and cumulative post counts across the 12-month platform observation period using SQLite window functions (`LAG`, `SUM() OVER`).

In [ ]:
import os
import sqlite3
import pandas as pd

# File Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(BASE_DIR, "data", "data_vortex.db")
SQL_PATH = os.path.join(BASE_DIR, "sql", "challenge_05_monthly_publishing_trends.sql")

print(f"Target Database: {DB_PATH}")
print(f"SQL Script:      {SQL_PATH}")

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print("Connected to SQLite database successfully.")

### 2. Monthly Publishing Trends & Window Functions
Executes the primary analytical query computing:
- `month` (`YYYY-MM`)
- `monthly_post_count`
- `month_over_month_change` (`LAG` difference; NULL for initial month)
- `month_over_month_change_pct` (MoM percentage change; NULL for initial month)
- `cumulative_post_count` (running total)

In [ ]:
q_monthly = """
WITH monthly_aggregation AS (
    SELECT 
        strftime('%Y-%m', timestamp) AS month,
        COUNT(post_id) AS monthly_post_count
    FROM posts
    GROUP BY strftime('%Y-%m', timestamp)
)
SELECT 
    month,
    monthly_post_count,
    monthly_post_count - LAG(monthly_post_count) OVER (ORDER BY month) AS month_over_month_change,
    ROUND(
        100.0 * (monthly_post_count - LAG(monthly_post_count) OVER (ORDER BY month)) 
        / LAG(monthly_post_count) OVER (ORDER BY month), 
        2
    ) AS month_over_month_change_pct,
    SUM(monthly_post_count) OVER (
        ORDER BY month 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_post_count
FROM monthly_aggregation
ORDER BY month ASC;
"""

df_monthly = pd.read_sql_query(q_monthly, conn)
df_monthly

### 3. Monthly Publishing Extremes
Identifies peak/trough months and largest MoM variations.

In [ ]:
q_extremes = """
WITH monthly_aggregation AS (
    SELECT 
        strftime('%Y-%m', timestamp) AS month,
        COUNT(post_id) AS monthly_post_count
    FROM posts
    GROUP BY strftime('%Y-%m', timestamp)
),
monthly_trends AS (
    SELECT 
        month,
        monthly_post_count,
        monthly_post_count - LAG(monthly_post_count) OVER (ORDER BY month) AS mom_change,
        ROUND(
            100.0 * (monthly_post_count - LAG(monthly_post_count) OVER (ORDER BY month)) 
            / LAG(monthly_post_count) OVER (ORDER BY month), 
            2
        ) AS mom_change_pct
    FROM monthly_aggregation
)
SELECT * FROM (
    SELECT 
        'Month with Highest Post Volume' AS metric, 
        month, 
        monthly_post_count AS metric_value, 
        NULL AS mom_change_pct
    FROM monthly_aggregation
    ORDER BY monthly_post_count DESC 
    LIMIT 1
)
UNION ALL
SELECT * FROM (
    SELECT 
        'Month with Lowest Post Volume' AS metric, 
        month, 
        monthly_post_count AS metric_value, 
        NULL AS mom_change_pct
    FROM monthly_aggregation
    ORDER BY monthly_post_count ASC 
    LIMIT 1
)
UNION ALL
SELECT * FROM (
    SELECT 
        'Largest Positive MoM Increase' AS metric, 
        month, 
        mom_change AS metric_value, 
        mom_change_pct
    FROM monthly_trends
    WHERE mom_change IS NOT NULL
    ORDER BY mom_change DESC 
    LIMIT 1
)
UNION ALL
SELECT * FROM (
    SELECT 
        'Largest Negative MoM Decrease' AS metric, 
        month, 
        mom_change AS metric_value, 
        mom_change_pct
    FROM monthly_trends
    WHERE mom_change IS NOT NULL
    ORDER BY mom_change ASC 
    LIMIT 1
);
"""

df_extremes = pd.read_sql_query(q_extremes, conn)
df_extremes

### 4. Validation Checks
Confirms exact reconciliation against data integrity criteria.

In [ ]:
# Validation 1: Total posts sum
sum_posts = df_monthly['monthly_post_count'].sum()
print(f"1. Monthly counts sum: {sum_posts} (Expected: 12000) -> {'PASS' if sum_posts == 12000 else 'FAIL'}")

# Validation 2: Number of months
num_months = len(df_monthly)
print(f"2. Number of months:    {num_months} (Expected: 12) -> {'PASS' if num_months == 12 else 'FAIL'}")

# Validation 3: Cumulative total ends at 12,000
final_cumulative = df_monthly['cumulative_post_count'].iloc[-1]
print(f"3. Final cumulative:    {final_cumulative} (Expected: 12000) -> {'PASS' if final_cumulative == 12000 else 'FAIL'}")

# Validation 4: First month MoM change is NaN / NULL
first_mom_is_null = pd.isna(df_monthly['month_over_month_change'].iloc[0])
print(f"4. First month is NULL: {first_mom_is_null} (Expected: True) -> {'PASS' if first_mom_is_null else 'FAIL'}")

# Validation 5: Database unchanged
db_posts = conn.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
print(f"5. DB post count:       {db_posts} (Expected: 12000) -> {'PASS' if db_posts == 12000 else 'FAIL'}")

In [ ]:
# Close connection
conn.close()
print("Database connection closed cleanly.")